In [6]:
import os
import pytesseract

# Force add Tesseract to PATH inside Jupyter
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Tesseract-OCR"

# Explicitly set executable location
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

print("Tesseract Version:", pytesseract.get_tesseract_version())

Tesseract Version: 5.5.0.20241111


In [7]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage


In [8]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "Finetuning_Final.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: Finetuning_Final.pdf


preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

✅ Extracted 460 elements


In [9]:
len(elements)

460

In [10]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [11]:
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()


Found 11 images


{'type': 'Image',
 'element_id': '4668559aebd1cdbf255772930065eda1',
 'text': 'Large Unla-  beled Corpus  Pre-training  Pre-trained Model  Small Labeled  Task Dataset  Fine-tuning  Fine-tuned Model ',
 'metadata': {'detection_class_prob': 0.8959733247756958,
  'coordinates': {'points': ((np.float64(577.0460205078125),
     np.float64(209.99917602539062)),
    (np.float64(577.0460205078125), np.float64(922.3873291015625)),
    (np.float64(1637.0665283203125), np.float64(922.3873291015625)),
    (np.float64(1637.0665283203125), np.float64(209.99917602539062))),
   'system': 'PixelSpace',
   'layout_width': 2205,
   'layout_height': 1241},
  'last_modified': '2026-02-21T23:03:39',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 3,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'filename': 'Finetuning_Final.pdf'}}

In [13]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[1].to_dict()

# Use https://jsfiddle.net/ to view the table html 

Found 2 tables


{'type': 'Table',
 'element_id': 'ce6ebad3a366ed0d6f075c5f435d6689',
 'text': 'Potentially highest Extremely memory-intensive. performance. Simple to implement. Creates a full model copy (many GB) for each task.',
 'metadata': {'detection_class_prob': 0.6531597375869751,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(74.8359375),
     np.float64(332.8877868652344)),
    (np.float64(74.8359375), np.float64(617.3003540039062)),
    (np.float64(2065.7109375), np.float64(617.3003540039062)),
    (np.float64(2065.7109375), np.float64(332.8877868652344))),
   'system': 'PixelSpace',
   'layout_width': 2205,
   'layout_height': 1241},
  'last_modified': '2026-02-21T23:03:39',
  'text_as_html': '<table><tbody><tr><td>performance.</td><td>@ Requires expensive hardware (e.g., A100 GPU</td></tr><tr><td>@ Simple to implement.</td><td>@ Creates a full model copy (many GB) for each</td></tr></tbody></table>',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number

In [14]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 21 chunks


In [16]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary using Ollama (phi3)"""

    try:
        # Initialize local LLM
        llm = ChatOllama(
            model="phi3",
            temperature=0
        )

        prompt_text = f"""
You are creating a searchable description for document retrieval.

TEXT CONTENT:
{text}

"""

        # Convert tables to readable text
        if tables:
            prompt_text += "\nTABLE DATA:\n"
            for i, table in enumerate(tables):
                prompt_text += f"\nTable {i+1} (HTML format):\n{table}\n"

        # Images cannot be analyzed by phi3
        if images:
            prompt_text += f"\nNOTE: This content contains {len(images)} image(s). " \
                           f"Describe likely visual elements based on context.\n"

        prompt_text += """

TASK:
Generate a detailed, searchable description that includes:

1. Key facts and numbers
2. Main topics discussed
3. Questions this content could answer
4. Important table insights
5. Alternative search keywords

Make it retrieval-optimized.

SEARCHABLE DESCRIPTION:
"""

        response = llm.invoke(prompt_text)

        return response.content if hasattr(response, "content") else response

    except Exception as e:
        print(f"❌ AI summary failed: {e}")

        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/21
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/21
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **Searchable Description for Document Retrieval on Fine-Tuning LLMs by Dr. Abhishek Kaushal (ACAI) - August 5, 2025**

1. **Key Facts and Numbers:** The document discusses the transition from a pre-tr...
   Processing chunk 3/21
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/21
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **Searchable Description for Document Retrieval on "Fine-Tuning LLMs" by Dr. Abhishek Kaushal (ACAI) - 

In [17]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "Dr. Abhishek Kaushal (ACAI)\\n\\nA Detailed Guide to Fine-Tuning GPT, BERT, & T5\\n\\nDr. Abhishek Kaushal\\n\\nACAI\\n\\nAugust 5, 2025\\n\\nFine-Tuning LLMs\\n\\nAugust 5, 2025\\n\\n1/39\\n\\nWhat is Fine-Tuning? The Core Idea I Fine-tuning is a transfer learning technique that adapts a general, pre-trained model to a specific downstream task.\\n\\nAnalogy: Think of a master chef (the pre-trained model) who knows thousands of cooking techniques, now learning a specific regional recipe (the fine-tuning task).\\n\\nIt\\u2019s a two-stage process:\\n\\n1 Pre-training: A model learns general language patterns from a massive text corpus. This step is computationally immense and done for us.\\n\\n2 Fine-tuning: We take the pre-trained model and train it further on a much smaller, task-specific dataset. This specializes the model\\u2019s knowledge.\\n\\nKey Idea\\n\\nWe don\\u2019t start from scratch. We leverage the powerful knowledge 

In [18]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 21 chunks to chunks_export.json


In [20]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store (Ollama Local Embeddings)"""
    print("🔮 Creating embeddings and storing in ChromaDB...")
        
    # Local embedding model via Ollama
    embedding_model = OllamaEmbeddings(
        model="nomic-embed-text"
    )
    
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    
    print("--- Finished creating vector store ---")
    print(f"✅ Vector store created and saved to {persist_directory}")
    
    return vectorstore


# Create the vector store
db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in ChromaDB...
--- Creating vector store ---
--- Finished creating vector store ---
✅ Vector store created and saved to dbv1/chroma_db


In [21]:
query = "What are the two main components of fine tuning? "
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 3 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': 'Dr. Abhishek Kaushal (ACAI)\n\nA Detailed Guide to Fine-Tuning GPT, BERT, & T5\n\nDr. Abhishek Kaushal\n\nACAI\n\nAugust 5, 2025\n\nFine-Tuning LLMs\n\nAugust 5, 2025\n\n1/39\n\nWhat is Fine-Tuning? The Core Idea I Fine-tuning is a transfer learning technique that adapts a general, pre-trained model to a specific downstream task.\n\nAnalogy: Think of a master chef (the pre-trained model) who knows thousands of cooking techniques, now learning a specific regional recipe (the fine-tuning task).\n\nIt’s a two-stage process:\n\n1 Pre-training: A model learns general language patterns from a massive text corpus. This step is computationally immense and done for us.\n\n2 Fine-tuning: We take the pre-trained model and train it further on a much smaller, task-specific dataset. This specializes the model’s knowledge.\n\nKey Idea\n\nWe don’t start from scratch. We leverage the powerful knowledge already encoded in a massive pre-trained model.\n\nDr. Abhish

In [22]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")
    
    print("🎉 Pipeline completed successfully!")
    return db

# Run the complete pipeline

In [25]:
db = run_complete_ingestion_pipeline("Finetuning_Final.pdf")

🚀 Starting RAG Ingestion Pipeline
📄 Partitioning document: Finetuning_Final.pdf
✅ Extracted 460 elements
🔨 Creating smart chunks...
✅ Created 21 chunks
🧠 Processing chunks with AI Summaries...
   Processing chunk 1/21
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 2/21
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     → AI summary created successfully
     → Enhanced content preview: **Searchable Description for Document Retrieval on Fine-Tuning LLMs by Dr. Abhishek Kaushal (ACAI) - August 5, 2025**

1. **Key Facts and Numbers:** The document discusses the transition from a pre-tr...
   Processing chunk 3/21
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 4/21
     Types found: ['image', 'text']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     → AI summary creat

In [28]:
# Query the vector store
query = "According to Document 2 Table 1(B), how does PEFT reduce VRAM usage compared to full fine-tuning?"

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOllama(
            model="phi3",
            temperature=0
        )

        
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks: 
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)

Based on Document 2 Table 1(B), PEFT (Parameter-efficient Fine-Tuning) reduces VRAM usage compared to full fine-tuning by not updating every single weight of the pre-trained model. Instead, it employs a technique like LoRA that makes gentle updates to certain parts of the model while rapidly training new task-specific components. This approach avoids creating large memory footprint models for each downstream task and mitigates catastrophic forgetting by preserving general language knowledge acquired during pre-training, as discussed in Document 2 Table 1(B) under "Pros" which lists reduced VRAM usage among its benefits compared to full fine-tuning.

Moreover, the table illustrates that a single parameter with AdamW optimizer's state requires about 12 bytes of GPU RAM (considering both weight and gradient). When applied to an entire model like BERT or GPT which might have billions of parameters as mentioned in Document 2 Table 1(B), the VRAM requirement for full fine-tuning would be ast